In [1]:
import re
from collections import defaultdict
from typing import List
from utils.exploit_gates1 import NetlistParser

from detection_codes.does_have_trojan0 import does_have_trojan0
from detection_codes.does_have_trojan4 import does_have_trojan4
from detection_codes.does_have_trojan5 import does_have_trojan5
from detection_codes.does_have_trojan2 import does_have_trojan2
from detection_codes.does_have_trojan1 import does_have_trojan1
from detection_codes.does_have_trojan7 import does_have_trojan7
from detection_codes.does_have_trojan6 import does_have_trojan6
from detection_codes.does_have_trojan3 import does_have_trojan3
from detection_codes.does_have_trojan9 import does_have_trojan9



In [2]:
def does_have_any_trojan(target_file) -> List:
    Trojan_number_found = -1
    [trojan_gates, does_exist] = does_have_trojan0(target_file)
    if does_exist:
        Trojan_number_found = 0
    if not does_exist:
        # print("Does not have trojan0 pattern")
        [trojan_gates, does_exist] = does_have_trojan1(target_file)
        if does_exist:
            Trojan_number_found = 1
        if not does_exist:
            # print("Does not have trojan1 pattern either")
            [trojan_gates, does_exist] = does_have_trojan2(target_file)
            if does_exist:
                Trojan_number_found = 2
            if not does_exist:
                # print("Does not have trojan2 pattern either")
                [trojan_gates, does_exist] = does_have_trojan4(target_file)
                if does_exist:
                    Trojan_number_found = 4
                if not does_exist:
                    # print("Does not have trojan4 pattern either")
                    [trojan_gates, does_exist] = does_have_trojan5(target_file)
                    if does_exist:
                        Trojan_number_found = 5
                    if not does_exist:
                        # print("Does not have trojan6 pattern either")
                        [trojan_gates, does_exist] = does_have_trojan7(target_file)
                        if does_exist:
                            Trojan_number_found = 7
                        if not does_exist:
                            # print("Does not have trojan7 pattern either")
                            [trojan_gates, does_exist] = does_have_trojan3(target_file)
                            if does_exist:
                                Trojan_number_found = 3
                            if not does_exist:
                                [trojan_gates, does_exist] = does_have_trojan6(target_file)
                                if does_exist:
                                    Trojan_number_found = 6
                                if not does_exist:
                                    [trojan_gates, does_exist] = does_have_trojan9(target_file)
                                    if does_exist:
                                        Trojan_number_found = 9                            
                                
    return [trojan_gates, does_exist, Trojan_number_found]

In [3]:
# Evaluation

from utils.Tokenizer_functions import extract_trojan_gates

final_score = 0
for Test_Design_Number in range(60):
    score = 0
    target_file = f"./data/release_hidden_0923/release_hidden/design{Test_Design_Number}.v"
    if Test_Design_Number >= 40:
        [trojan_gates, does_exist, Trojan_number_found] = does_have_any_trojan(target_file)
        if(does_exist):
            score = 0
        else:
            score = 2
        parser = NetlistParser()
        parser.parse_netlist(target_file)
        final_score = final_score + score
        print_excel_outputs = [Test_Design_Number, 0, len(trojan_gates), 0, len(parser.gates) - len(trojan_gates), 0, 0, 0, score, Trojan_number_found if Trojan_number_found != -1 else 'N/A']
        print('\t'.join(map(str, print_excel_outputs)))
        continue

    trojan_gates = []

    parser = NetlistParser()
    parser.parse_netlist(target_file)
    [trojan_gates, does_exist, Trojan_number_found] = does_have_any_trojan(target_file)
    reference_Trojans_file = f"./data/release_hidden_0923/release_hidden/result{Test_Design_Number}.txt"
    actual_trojan_gates = extract_trojan_gates(reference_Trojans_file)
    if not does_exist:
        score = 0
        print_excel_outputs = [Test_Design_Number, 0, 0, len(actual_trojan_gates), len(parser.gates) - len(actual_trojan_gates), 0, 0, 0, score]
        print('\t'.join(map(str, print_excel_outputs)))
        continue
    score = 2


    true_positive = 0
    for gate in parser.gates:
        if gate in trojan_gates and gate in actual_trojan_gates:
            true_positive = true_positive + 1


    false_positive = 0
    for gate in trojan_gates:
        if gate not in actual_trojan_gates:
            false_positive = false_positive + 1

 
    false_negative = 0
    for gate in actual_trojan_gates:
        if gate not in trojan_gates:
            false_negative = false_negative + 1

    true_negative = 0
    for gate in parser.gates:
        if gate not in trojan_gates and gate not in actual_trojan_gates:
            true_negative = true_negative + 1


    TPR = true_positive / len(actual_trojan_gates) if actual_trojan_gates else 0
    FPR = false_positive / (len(actual_trojan_gates) + false_negative) if (len(actual_trojan_gates) + false_negative) > 0 else 0
    precision = true_positive / (true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
    recall = true_positive / (true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    f1_score = 2 * (recall * precision) / (recall + precision) if (recall + precision) > 0 else 0

    score = score + f1_score
    final_score = final_score + score

    print_excel_outputs = [Test_Design_Number,true_positive, false_positive, false_negative, true_negative, precision, TPR, f1_score, score]
    if (Trojan_number_found != -1):
        print_excel_outputs.append(Trojan_number_found)
    print('\t'.join(map(str, print_excel_outputs)))

#print total score
print(f"Final Score: {final_score:.4f}")


0	71	0	0	265	1.0	1.0	1.0	3.0	0
1	71	0	0	636	1.0	1.0	1.0	3.0	0
2	22	0	1	846	1.0	0.9565217391304348	0.9777777777777777	2.977777777777778	1
3	22	0	1	630	1.0	0.9565217391304348	0.9777777777777777	2.977777777777778	1
4	24	0	1	1471	1.0	0.96	0.9795918367346939	2.979591836734694	2
5	24	0	1	1271	1.0	0.96	0.9795918367346939	2.979591836734694	2
6	24	0	1	1471	1.0	0.96	0.9795918367346939	2.979591836734694	2
7	128	0	0	1083	1.0	1.0	1.0	3.0	3
8	147	0	1	1163	1.0	0.9932432432432432	0.9966101694915254	2.9966101694915253	4
9	147	0	1	1163	1.0	0.9932432432432432	0.9966101694915254	2.9966101694915253	4
10	41	1	0	1150	0.9761904761904762	1.0	0.9879518072289156	2.9879518072289155	5
11	41	1	0	1150	0.9761904761904762	1.0	0.9879518072289156	2.9879518072289155	5
12	42	0	0	1192	1.0	1.0	1.0	3.0	6
13	42	0	0	1192	1.0	1.0	1.0	3.0	6
14	85	0	0	1162	1.0	1.0	1.0	3.0	7
15	85	0	0	1162	1.0	1.0	1.0	3.0	7
16	5068	0	0	1162	1.0	1.0	1.0	3.0	9
17	5068	0	0	1162	1.0	1.0	1.0	3.0	9
18	2002	0	0	1162	1.0	1.0	1.0	3.0	9
19	2002	0	0	1162	1.0